In [1]:
import pandas as pd
import re
from hazm import Normalizer, word_tokenize, stopwords_list
import torch  
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

D:\ElmosYar\ElmosYar\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [25]:
df = pd.read_csv("../full_cleaned_reviews.csv")

the following cell does not give me clean data comments col is better than clean_comments !!!!

In [3]:
normalizer = Normalizer()
stopwords = stopwords_list()

def preprocess_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    # Normalize Persian text
    text = normalizer.normalize(text)
    # Remove punctuation and non-word characters
    text = re.sub(r'[^\w\s]', '', text)  # Removes punctuation
    # Tokenize
    tokens = word_tokenize(text)
    # Remove stopwords
    tokens = [t for t in tokens if t not in stopwords]
    return " ".join(tokens)

# Apply preprocessing to the 'comments' column
df['cleaned_comments'] = df['comments'].apply(preprocess_text)

In [7]:
base_model_name = "m3hrdadfi/albert-fa-base-v2"
sentiment_model_name = "m3hrdadfi/albert-fa-base-v2-sentiment-multi"

tokenizer = AutoTokenizer.from_pretrained(base_model_name)
model = AutoModelForSequenceClassification.from_pretrained(sentiment_model_name)

# CORRECT: Remove top_k=None → returns only the TOP label
sentiment_pipeline = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    truncation=True,
    max_length=512,
    device=-1  # CPU,
)

# Test — should now return single dict
print(sentiment_pipeline("عالی بود"))     # → [{'label': 'Positive', 'score': 0.99}]
print(sentiment_pipeline("افتضاح بود"))  # → [{'label': 'Negative', 'score': 0.99}]
print(sentiment_pipeline("معمولی"))      # → [{'label': 'Neutral', 'score': 0.98}]

# Label mapping (human readable)
label_to_score = {
    'Positive':  1.0,
    'Negative': -1.0,
    'Neutral':   0.0
}

def model_sentiment(text):
    if not text.strip():
        return 0.0
    # Now returns list with ONE dict → take [0]
    result = sentiment_pipeline(text)[0]
    label = result['label']
    score = result['score']
    return round(label_to_score[label] * score, 4)

print("Running sentiment analysis on all reviews...")
df['sentiment_score'] = df['cleaned_comments'].apply(model_sentiment)

df['sentiment_label'] = df['sentiment_score'].apply(
    lambda x: 'Positive' if x > 0.2 else ('Negative' if x < -0.2 else 'Neutral')
)

df.to_csv('public/data/reviews_sentiment_analysis.csv', index=False, encoding='utf-8-sig')

print("All done! First 10 results:")
print(df[['comments', 'sentiment_score', 'sentiment_label']].head(10))


# # # Apply (this will now work perfectly)
# # print("Running sentiment analysis on all reviews...")
# # df['sentiment_score'] = df
# # ['clean_comments'].apply(model_sentiment)

# # # Optional: Add readable label column
# # df['sentiment_label'] = df
# # ['sentiment_score'].apply(
# #     lambda x: 'Positive' if x > 0.2 else ('Negative' if x < -0.2 else 'Neutral')
# # )

# print("Running sentiment analysis on all reviews...")
# df['sentiment_score'] = df['clean_comments'].apply(model_sentiment)

# df['sentiment_label'] = df['sentiment_score'].apply(
#     lambda x: 'Positive' if x > 0.2 else ('Negative' if x < -0.2 else 'Neutral')
# )

# # Save the results
# df.to_csv('public/data/reviews_sentiment_analysis.csv', index=False, encoding='utf-8-sig')
# # Or if you want a different name/location:
# # df.to_csv('sentiment_analysis_results.csv', index=False, encoding='utf-8-sig')

# print("All done! First 10 results:")
# print(df[['comments', 'sentiment_score', 'sentiment_label']].head(10))
# # Save
# # df.to_csv('sentiment_analysis_results.csv', index=False, encoding='utf-8-sig')

print("All done! First 10 results:")
print(df
[['comments', 'sentiment_score', 'sentiment_label']].head(10))

Device set to use cpu


[{'label': 'Positive', 'score': 0.9966872334480286}]
[{'label': 'Negative', 'score': 0.9946185350418091}]
[{'label': 'Neutral', 'score': 0.984097421169281}]
Running sentiment analysis on all reviews...
All done! First 10 results:
                                            comments  sentiment_score  \
0                                چیزی اضافه ایی نیست           0.9584   
1  خوب درس نمیده اصلا، نمره ها رو خوب نمیده و اصل...           0.0000   
2  در کل اگر که دنبال یک استاد با ادب با دانشجو م...           0.0000   
3                                  سخت گیر و پربازده           0.0000   
4                                چیزی اضافه ایی نیست           0.9584   
5  دقت کنید هر جلسه مطالب را بخونید ارشد شب امتحا...           0.7083   
6                                چیزی اضافه ایی نیست           0.9584   
7  اعصابی براتون نمیمونه چه نمره خوبی بگیرید چه ب...           0.0000   
8  استاد خوبی هستند فقط امتحان های سختی میگیرن که...           0.8456   
9             سعی کنید سر کلاس خوب گوش ب

In [9]:
df.head(40)

,professor_name_raw,department,course_name,sources,attendance,exam_sufficient,grading,rating_1,rating_2,rating_3,...,rating_6,contact,term,comments,clean_key,professor_id,professor_name_standard,cleaned_comments,sentiment_score,sentiment_label
0,سید محمد شهرتاش,برق,بررسی ۱-بررسی۲-حفاظت,فایل پاورپوینت یا پی دی اف می دهد,حضور و غیاب نمی کند,بله,منصفانه و هرچی خودت بگیری,8.0,9.0,7.0,...,10.0,چیزی ندارم,مهر 99,چیزی اضافه ایی نیست,محمد شهرتاش,423,محمد شهرتاش,اضافه ایی,0.9584,Positive
1,دکتر هاجر فلاحتی,مهندسی_کامپیوتر,مدار منطقی- طراحی سیستم های کامپیوتری,یادم نمیاد,حضور مهم نیست اما تاثیر مثبت دارد,خیر,نمره خوبی نمیشه ازشون گرفت,1.0,2.0,1.0,...,10.0,چیزی ندارم,بهمن 98,خوب درس نمیده اصلا، نمره ها رو خوب نمیده و اصل...,هاجر فلاحتی,424,هاجر فلاحتی,درس نمیده اصلا نمرهها نمیده اصلا ترم اخر ترم ج...,0.0000,Neutral
2,علم الهدی,مهندسی_مواد,بلور شناسی,فایل پاورپوینت یا پی دی اف می دهد,حضور و غیاب نمی کند,بله,منصفانه و هرچی خودت بگیری,9.0,8.0,10.0,...,10.0,بله,بهمن 99,در کل اگر که دنبال یک استاد با ادب با دانشجو م...,علم الهدی,234,سمیه اعلم الهدی,دنبال استاد ادب دانشجو میخواید گزینه خوبیه,0.0000,Neutral
3,حجت قاسمی,مکانیک,سوخت و احتراق,یک کتاب مرجع معرفی می کند,حضور و غیاب نمی کند,بله,منصفانه و هرچی خودت بگیری,10.0,10.0,9.0,...,9.0,شماره تماس,مهر 98,سخت گیر و پربازده,حجت قاسمی,578,حجت قاسمی,سختگیر پربازده,0.0000,Neutral
4,فریبرز قاسمی,مهندسی_کامپیوتر,برنامه نویسی کامپیوتر,فایل پاورپوینت یا پی دی اف می دهد,حضور و غیاب نمی کند,بله,منصفانه و هرچی خودت بگیری,8.0,10.0,8.0,...,9.0,چیزی ندارم,بهمن 99,چیزی اضافه ایی نیست,فریبرز قاسمی,19,فریبرز قاسمی فیض آبادی,اضافه ایی,0.9584,Positive
5,محمد باقر صادقی,مهندسی_مواد,درس های پلاستیسیته و مکانیک محیط های پیوسته,فایل پاورپوینت یا پی دی اف می دهد,حضور مهم نیست اما تاثیر مثبت دارد,بله,منصفانه و هرچی خودت بگیری,10.0,10.0,10.0,...,10.0,بله ایشان گروه مجازی جهت ارتباط علمی با دانشجو...,مهر 99,دقت کنید هر جلسه مطالب را بخونید ارشد شب امتحا...,محمد باقر صادقی,192,محمد باقر صادقی,دقت جلسه مطالب بخونید ارشد شب امتحانی ترمی هست...,0.7083,Positive
6,هادی خرمی شاد,مکانیک,اتصالات چسبی,در حین تدریس جزوه می نویسیم,حضور مهم نیست اما تاثیر مثبت دارد,بله,منصفانه و هرچی خودت بگیری,10.0,10.0,10.0,...,10.0,چیزی ندارم,بهمن 98,چیزی اضافه ایی نیست,هادی خرمی شاد,311,هادی خرمی شاد,اضافه ایی,0.9584,Positive
7,محمدعظیم کرمی,برق,الکترونیک 1,فایل پاورپوینت یا پی دی اف می دهد,حضور مهم است و تاثیر مستقیم دارد,بله,نمره خوبی نمیشه ازشون گرفت,5.0,7.0,8.0,...,5.0,چیزی ندارم,بهمن 99,اعصابی براتون نمیمونه چه نمره خوبی بگیرید چه ب...,محمدعظیم کرمی,292,محمدعظیم کرمی,اعصابی براتون نمیمونه نمره بگیرید بد ترم ۴۰ کل...,0.0000,Neutral
8,سید علیرضا نظام الحسینی,برق,سیگنالها و سیستمها,کتاب مرجع معرفی میکند پاورپوینت هم میدهد اما ا...,حضور و غیاب نمی کند,بله,منصفانه و هرچی خودت بگیری,10.0,10.0,10.0,...,10.0,ایمیل ایشان بود که در سایت نیز هست.,مهر 99,استاد خوبی هستند فقط امتحان های سختی میگیرن که...,علیرضا نظام الحسینی,1,علیرضا نظام الحسینی عز آبادی,استاد امتحانهای سختی میگیرن اون خونده باشی میش...,0.8456,Positive
9,علی رضا ایوانی,مهندسی_مواد,درس شکل پذیری فلزات و شکست,فایل پاورپوینت یا پی دی اف می دهد,حضور مهم نیست اما تاثیر مثبت دارد,بله,منصفانه و هرچی خودت بگیری,10.0,10.0,10.0,...,10.0,بله گروه مجازی دارن,بهمن 99,سعی کنید سر کلاس خوب گوش بدید به مطالب,علی رضا ایوانی,333,علی رضا ایوانی,سر کلاس گوش بدید مطالب,0.9746,Positive


In [6]:
print(df.columns.tolist())


['professor_name_raw', 'department', 'course_name', 'sources', 'attendance', 'exam_sufficient', 'grading', 'rating_1', 'rating_2', 'rating_3', 'rating_4', 'rating_5', 'rating_6', 'contact', 'term', 'comments', 'clean_key', 'professor_id', 'professor_name_standard', 'cleaned_comments']


In [8]:
test_df

,professor_name_raw,department,course_name,sources,attendance,exam_sufficient,grading,rating_1,rating_2,rating_3,rating_4,rating_5,rating_6,contact,term,comments,sentiment_score,sentiment_label
0,سید محمد شهرتاش,برق,بررسی ۱-بررسی۲-حفاظت,فایل پاورپوینت یا پی دی اف می دهد,حضور و غیاب نمی کند,بله,منصفانه و هرچی خودت بگیری,8.0,9.0,7.0,9.0,8.0,10.0,چیزی ندارم,مهر 99,چیزی اضافه ایی نیست,0.0000,Neutral
1,دکتر هاجر فلاحتی,مهندسی_کامپیوتر,مدار منطقی- طراحی سیستم های کامپیوتری,یادم نمیاد,حضور مهم نیست اما تاثیر مثبت دارد,خیر,نمره خوبی نمیشه ازشون گرفت,1.0,2.0,1.0,3.0,2.0,10.0,چیزی ندارم,بهمن 98,خوب درس نمیده اصلا، نمره ها رو خوب نمیده و اصل...,-0.7150,Negative
2,علم الهدی,مهندسی_مواد,بلور شناسی,فایل پاورپوینت یا پی دی اف می دهد,حضور و غیاب نمی کند,بله,منصفانه و هرچی خودت بگیری,9.0,8.0,10.0,9.0,10.0,10.0,بله,بهمن 99,در کل اگر که دنبال یک استاد با ادب با دانشجو م...,0.0000,Neutral
3,حجت قاسمی,مکانیک,سوخت و احتراق,یک کتاب مرجع معرفی می کند,حضور و غیاب نمی کند,بله,منصفانه و هرچی خودت بگیری,10.0,10.0,9.0,9.0,9.0,9.0,شماره تماس,مهر 98,سخت گیر و پربازده,0.0000,Neutral
4,فریبرز قاسمی,مهندسی_کامپیوتر,برنامه نویسی کامپیوتر,فایل پاورپوینت یا پی دی اف می دهد,حضور و غیاب نمی کند,بله,منصفانه و هرچی خودت بگیری,8.0,10.0,8.0,10.0,7.0,9.0,چیزی ندارم,بهمن 99,چیزی اضافه ایی نیست,0.0000,Neutral
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,دکتر علی صدر,برق,مدار منطقی,در حین تدریس جزوه می نویسیم,حضور مهم نیست اما تاثیر مثبت دارد,تا حدودی,منصفانه و هرچی خودت بگیری,10.0,10.0,10.0,8.0,8.0,9.0,ایمیل ، تلگرام و lms,بهمن 99,دکتر صدر باقلم نوری تدریس میکنن و همه ی مطالب ...,0.7337,Positive
96,دکتر سید محمد شهرتاش,برق,مدارهای الکتریکی ۲,فایل پاورپوینت یا پی دی اف می دهد,حضور و غیاب نمی کند,خیر,نمره خوبی نمیشه ازشون گرفت,4.0,5.0,3.0,3.0,5.0,8.0,خیر,بهمن 99,هیچ منبعی معرفی نمیکنن و طبق هیچ کتابی درس نمی...,-0.9885,Negative
97,رضا پارچه باف,معارف,اخلاق,یک کتاب مرجع معرفی می کند,حضور مهم است و تاثیر مستقیم دارد,بله,منصفانه و هرچی خودت بگیری,10.0,10.0,10.0,10.0,10.0,10.0,چیزی ندارم,بهمن 98,جا برای گرفتن نمره اضافه زیاده\nتحقیق و ارائه ...,0.8361,Positive
98,دکتر سید ادیب ابریشمی فر,برق,الکترونیک ۱,از جزوات ترم های گذشته استفاده می کند,حضور و غیاب نمی کند,تا حدودی,دست باز و با ارفاق,10.0,10.0,9.0,10.0,9.0,10.0,ایمیل,بهمن 99,دکتر ابریشمی فر از روی جزواتی که دانشجوهای ترم...,0.0000,Neutral


In [11]:
df=pd.read_csv('extracted_reviews_v3_final.csv')

In [16]:
df

,professor_name_raw,department,course_name,sources,attendance,exam_sufficient,grading,rating_1,rating_2,rating_3,rating_4,rating_5,rating_6,contact,term,comments,clean_key,professor_id,professor_name_standard
0,سید محمد شهرتاش,برق,بررسی ۱-بررسی۲-حفاظت,فایل پاورپوینت یا پی دی اف می دهد,حضور و غیاب نمی کند,بله,منصفانه و هرچی خودت بگیری,8.0,9.0,7.0,9.0,8.0,10.0,چیزی ندارم,مهر 99,چیزی اضافه ایی نیست,محمد شهرتاش,423,محمد شهرتاش
1,دکتر هاجر فلاحتی,مهندسی_کامپیوتر,مدار منطقی- طراحی سیستم های کامپیوتری,یادم نمیاد,حضور مهم نیست اما تاثیر مثبت دارد,خیر,نمره خوبی نمیشه ازشون گرفت,1.0,2.0,1.0,3.0,2.0,10.0,چیزی ندارم,بهمن 98,خوب درس نمیده اصلا، نمره ها رو خوب نمیده و اصل...,هاجر فلاحتی,424,هاجر فلاحتی
2,علم الهدی,مهندسی_مواد,بلور شناسی,فایل پاورپوینت یا پی دی اف می دهد,حضور و غیاب نمی کند,بله,منصفانه و هرچی خودت بگیری,9.0,8.0,10.0,9.0,10.0,10.0,بله,بهمن 99,در کل اگر که دنبال یک استاد با ادب با دانشجو م...,علم الهدی,234,سمیه اعلم الهدی
3,حجت قاسمی,مکانیک,سوخت و احتراق,یک کتاب مرجع معرفی می کند,حضور و غیاب نمی کند,بله,منصفانه و هرچی خودت بگیری,10.0,10.0,9.0,9.0,9.0,9.0,شماره تماس,مهر 98,سخت گیر و پربازده,حجت قاسمی,578,حجت قاسمی
4,فریبرز قاسمی,مهندسی_کامپیوتر,برنامه نویسی کامپیوتر,فایل پاورپوینت یا پی دی اف می دهد,حضور و غیاب نمی کند,بله,منصفانه و هرچی خودت بگیری,8.0,10.0,8.0,10.0,7.0,9.0,چیزی ندارم,بهمن 99,چیزی اضافه ایی نیست,فریبرز قاسمی,19,فریبرز قاسمی فیض آبادی
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,دکتر علی صدر,برق,مدار منطقی,در حین تدریس جزوه می نویسیم,حضور مهم نیست اما تاثیر مثبت دارد,تا حدودی,منصفانه و هرچی خودت بگیری,10.0,10.0,10.0,8.0,8.0,9.0,ایمیل ، تلگرام و lms,بهمن 99,دکتر صدر باقلم نوری تدریس میکنن و همه ی مطالب ...,علی صدر,621,علی صدر
96,دکتر سید محمد شهرتاش,برق,مدارهای الکتریکی ۲,فایل پاورپوینت یا پی دی اف می دهد,حضور و غیاب نمی کند,خیر,نمره خوبی نمیشه ازشون گرفت,4.0,5.0,3.0,3.0,5.0,8.0,خیر,بهمن 99,هیچ منبعی معرفی نمیکنن و طبق هیچ کتابی درس نمی...,محمد شهرتاش,423,محمد شهرتاش
97,رضا پارچه باف,معارف,اخلاق,یک کتاب مرجع معرفی می کند,حضور مهم است و تاثیر مستقیم دارد,بله,منصفانه و هرچی خودت بگیری,10.0,10.0,10.0,10.0,10.0,10.0,چیزی ندارم,بهمن 98,جا برای گرفتن نمره اضافه زیاده\nتحقیق و ارائه ...,رضا پارچه باف,8,پورمدنی به جای پارچه باف
98,دکتر سید ادیب ابریشمی فر,برق,الکترونیک ۱,از جزوات ترم های گذشته استفاده می کند,حضور و غیاب نمی کند,تا حدودی,دست باز و با ارفاق,10.0,10.0,9.0,10.0,9.0,10.0,ایمیل,بهمن 99,دکتر ابریشمی فر از روی جزواتی که دانشجوهای ترم...,ادیب ابریشمی فر,197,ادیب ابریشمی فر


In [26]:
# Convert all comments to string
df['comments'] = df['comments'].astype(str)

def model_sentiment(text):
    if not isinstance(text, str):
        return 0.0

    text = text.strip()
    if not text:
        return 0.0
    
    result = sentiment_pipeline(text)[0]
    label = result['label']
    score = result['score']
    return round(label_to_score[label] * score, 4)

print("Running sentiment analysis on all reviews...")
df['sentiment_score'] = df['comments'].apply(model_sentiment)

df['sentiment_label'] = df['sentiment_score'].apply(
    lambda x: 'Positive' if x > 0.2 else ('Negative' if x < -0.2 else 'Neutral')
)

print(df[['comments', 'sentiment_score', 'sentiment_label']].head(10))


Running sentiment analysis on all reviews...
                                            comments  sentiment_score  \
0                                چیزی اضافه ایی نیست           0.0000   
1  خوب درس نمیده اصلا، نمره ها رو خوب نمیده و اصل...          -0.7150   
2  در کل اگر که دنبال یک استاد با ادب با دانشجو م...           0.0000   
3                                  سخت گیر و پربازده           0.0000   
4                                چیزی اضافه ایی نیست           0.0000   
5  دقت کنید هر جلسه مطالب را بخونید ارشد شب امتحا...           0.0000   
6                                چیزی اضافه ایی نیست           0.0000   
7  اعصابی براتون نمیمونه چه نمره خوبی بگیرید چه ب...          -0.9499   
8  استاد خوبی هستند فقط امتحان های سختی میگیرن که...           0.9579   
9             سعی کنید سر کلاس خوب گوش بدید به مطالب           0.0000   

  sentiment_label  
0         Neutral  
1        Negative  
2         Neutral  
3         Neutral  
4         Neutral  
5         Neutral  
6         N

In [27]:
df.tail(50)

,professor_name_raw,department,course_name,sources,attendance,exam_sufficient,grading,rating_1,rating_2,rating_3,...,rating_5,rating_6,contact,term,comments,clean_key,professor_id,professor_name_standard,sentiment_score,sentiment_label
4498,هادی صاحبی شاهم آبادی,مهندسی_صنایع,اقتصادمهندسی,در حین تدریس جزوه می نویسیم,حضور مهم است و تاثیر مستقیم دارد,تا حدودی,دست باز و با ارفاق,1.0,1.0,2.0,...,1.0,10.0,چیزی ندارم,بهمن403,اگر قراره جیزی یاد بگیرین ابدا با ایشون برندار...,هادی صاحبی شاهم آبادی,30,هادی شاهم آبادی صاحبی,-0.9960,Negative
4499,مریم بحرینی,فیزیک,فیزیک ۱,فایل پاورپوینت یا پی دی اف می دهد,حضور مهم است و تاثیر مستقیم دارد,خیر,نمره خوبی نمیشه ازشون گرفت,2.0,2.0,3.0,...,2.0,1.0,ایمیل,۴۰۳۲,مواظب باشید گیر ادم عقده ای نیافتید که حالتونو...,مریم بحرینی,462,مریم بحرینی,0.0000,Neutral
4500,احمد رضا فروغ,ریاضی,ریاضی ۲,فایل پاورپوینت یا پی دی اف می دهد,حضور مهم است و تاثیر مستقیم دارد,تا حدودی,نمره خوبی نمیشه ازشون گرفت,2.0,7.0,1.0,...,3.0,2.0,بله آیدی تلگرام و ایمیل,بهمن ۴۰۳,خود استاد جزوه های دست نویس شون رو توی کانالشو...,احمد رضا فروغ,371,احمد رضا فروغ,-0.9917,Negative
4501,امید وحیدی,مهندسی_شیمی,مبانی برنامه نویسی کامپیوتر,فایل پاورپوینت یا پی دی اف می دهد,حضور مهم نیست اما تاثیر مثبت دارد,خیر,منصفانه و هرچی خودت بگیری,6.0,7.0,6.0,...,9.0,10.0,@DrVahidi,بهمن 1403,پایتون تدریس میکنن که تسلط کافی ندارن و باید خ...,امید وحیدی,512,جواد وحیدی,0.0000,Neutral
4502,سمیه اعلم الهدی,مهندسی_مواد,بلورشناسی,فایل پاورپوینت یا پی دی اف می دهد,حضور مهم است و تاثیر مستقیم دارد,بله,منصفانه و هرچی خودت بگیری,10.0,10.0,10.0,...,10.0,10.0,ایمیل,بهمن ۱۴۰۳,استاد بسیار محترم و خوبی هستند\nاگر سر کلاس گو...,سمیه اعلم الهدی,234,سمیه اعلم الهدی,0.7421,Positive
4503,الهام فتاحی,ریاضی,ریاضی مهندسی,جزوه خودشون,حضور مهم نیست اما تاثیر مثبت دارد,بله,منصفانه و هرچی خودت بگیری,10.0,10.0,10.0,...,10.0,10.0,ایمیل,مهر 1402,جزوه ای که دارن برای امتحان خیلی خوبه ، تمرین ...,الهام فتاحی,428,الهام فتاحی,-0.5768,Negative
4504,حمید صفاری,مکانیک,ترمودینامیک 1,یک کتاب مرجع معرفی می کند,حضور مهم است و تاثیر مستقیم دارد,بله,منصفانه و هرچی خودت بگیری,8.0,8.0,8.0,...,6.0,8.0,چیزی ندارم,مهر 1403,اول ترم پرسه که امتحان رو فصل به فصل بگیرم یا ...,حمید صفاری,174,حمید صفاری نطنزی,-0.9542,Negative
4505,محمد حسینی,ریاضی,معادلات دیفرانسیل,یک کتاب مرجع معرفی می کند,حضور مهم نیست اما تاثیر مثبت دارد,بله,دست باز و با ارفاق,9.0,10.0,10.0,...,10.0,10.0,گروه بله,بهمن 403,بینهایت استاد فهمیده ای هستند و به راه های مخت...,محمد حسینی,58,محمد کاظم حسینی پور,0.9995,Positive
4506,علی عبدالعالی,برق,مدارهای الکتریکی یک,در حین تدریس جزوه می نویسیم,حضور و غیاب نمی کند,بله,نمره خوبی نمیشه ازشون گرفت,8.0,10.0,8.0,...,1.0,3.0,بله,بهمن 406,چیزی اضافه ایی نیست,علی عبدالعالی,369,علی عبدالعالی,0.0000,Neutral
4507,جمال الدین شریعتی,تربیت_بدنی,تربیت بدنی,فایل پاورپوینت یا پی دی اف می دهد,حضور مهم است و تاثیر مستقیم دارد,بله,منصفانه و هرچی خودت بگیری,10.0,10.0,10.0,...,10.0,10.0,شماره,مهر 403,خیلی استاد سختگیری هستند و راحت نمره نمیدن\nام...,جمال‌الدین شریعتی,112,جمال‌الدین شریعتی,0.0000,Neutral


In [28]:
len(df)

4548

In [23]:
df.to_csv('reviews_sentiment_analysis.csv', index=False, encoding='utf-8-sig')

In [13]:
# Convert all comments to string
df['comments'] = df['comments'].astype(str)

def model_sentiment(text):
    if not isinstance(text, str):
        return 0.0

    text = text.strip()
    if not text:
        return 0.0
    
    result = sentiment_pipeline(text)[0]
    label = result['label']
    score = result['score']
    return round(label_to_score[label] * score, 4)

print("Running sentiment analysis on all reviews...")
df['sentiment_score'] = df['comments'].apply(model_sentiment)

df['sentiment_label'] = df['sentiment_score'].apply(
    lambda x: 'Positive' if x > 0.2 else ('Negative' if x < -0.2 else 'Neutral')
)

print(df[['comments', 'sentiment_score', 'sentiment_label']].head(10))


Running sentiment analysis on all reviews...
                                            comments  sentiment_score  \
0                                چیزی اضافه ایی نیست           0.0000   
1  خوب درس نمیده اصلا، نمره ها رو خوب نمیده و اصل...          -0.7150   
2  در کل اگر که دنبال یک استاد با ادب با دانشجو م...           0.0000   
3                                  سخت گیر و پربازده           0.0000   
4                                چیزی اضافه ایی نیست           0.0000   
5  دقت کنید هر جلسه مطالب را بخونید ارشد شب امتحا...           0.0000   
6                                چیزی اضافه ایی نیست           0.0000   
7  اعصابی براتون نمیمونه چه نمره خوبی بگیرید چه ب...          -0.9499   
8  استاد خوبی هستند فقط امتحان های سختی میگیرن که...           0.9579   
9             سعی کنید سر کلاس خوب گوش بدید به مطالب           0.0000   

  sentiment_label  
0         Neutral  
1        Negative  
2         Neutral  
3         Neutral  
4         Neutral  
5         Neutral  
6         N

In [15]:
df.head(40)

,professor_name_raw,department,course_name,sources,attendance,exam_sufficient,grading,rating_1,rating_2,rating_3,rating_4,rating_5,rating_6,contact,term,comments,sentiment_score,sentiment_label
0,سید محمد شهرتاش,برق,بررسی ۱-بررسی۲-حفاظت,فایل پاورپوینت یا پی دی اف می دهد,حضور و غیاب نمی کند,بله,منصفانه و هرچی خودت بگیری,8.0,9.0,7.0,9.0,8.0,10.0,چیزی ندارم,مهر 99,چیزی اضافه ایی نیست,0.0000,Neutral
1,دکتر هاجر فلاحتی,مهندسی_کامپیوتر,مدار منطقی- طراحی سیستم های کامپیوتری,یادم نمیاد,حضور مهم نیست اما تاثیر مثبت دارد,خیر,نمره خوبی نمیشه ازشون گرفت,1.0,2.0,1.0,3.0,2.0,10.0,چیزی ندارم,بهمن 98,خوب درس نمیده اصلا، نمره ها رو خوب نمیده و اصل...,-0.7150,Negative
2,علم الهدی,مهندسی_مواد,بلور شناسی,فایل پاورپوینت یا پی دی اف می دهد,حضور و غیاب نمی کند,بله,منصفانه و هرچی خودت بگیری,9.0,8.0,10.0,9.0,10.0,10.0,بله,بهمن 99,در کل اگر که دنبال یک استاد با ادب با دانشجو م...,0.0000,Neutral
3,حجت قاسمی,مکانیک,سوخت و احتراق,یک کتاب مرجع معرفی می کند,حضور و غیاب نمی کند,بله,منصفانه و هرچی خودت بگیری,10.0,10.0,9.0,9.0,9.0,9.0,شماره تماس,مهر 98,سخت گیر و پربازده,0.0000,Neutral
4,فریبرز قاسمی,مهندسی_کامپیوتر,برنامه نویسی کامپیوتر,فایل پاورپوینت یا پی دی اف می دهد,حضور و غیاب نمی کند,بله,منصفانه و هرچی خودت بگیری,8.0,10.0,8.0,10.0,7.0,9.0,چیزی ندارم,بهمن 99,چیزی اضافه ایی نیست,0.0000,Neutral
5,محمد باقر صادقی,مهندسی_مواد,درس های پلاستیسیته و مکانیک محیط های پیوسته,فایل پاورپوینت یا پی دی اف می دهد,حضور مهم نیست اما تاثیر مثبت دارد,بله,منصفانه و هرچی خودت بگیری,10.0,10.0,10.0,10.0,10.0,10.0,بله ایشان گروه مجازی جهت ارتباط علمی با دانشجو...,مهر 99,دقت کنید هر جلسه مطالب را بخونید ارشد شب امتحا...,0.0000,Neutral
6,هادی خرمی شاد,مکانیک,اتصالات چسبی,در حین تدریس جزوه می نویسیم,حضور مهم نیست اما تاثیر مثبت دارد,بله,منصفانه و هرچی خودت بگیری,10.0,10.0,10.0,10.0,10.0,10.0,چیزی ندارم,بهمن 98,چیزی اضافه ایی نیست,0.0000,Neutral
7,محمدعظیم کرمی,برق,الکترونیک 1,فایل پاورپوینت یا پی دی اف می دهد,حضور مهم است و تاثیر مستقیم دارد,بله,نمره خوبی نمیشه ازشون گرفت,5.0,7.0,8.0,8.0,6.0,5.0,چیزی ندارم,بهمن 99,اعصابی براتون نمیمونه چه نمره خوبی بگیرید چه ب...,-0.9499,Negative
8,سید علیرضا نظام الحسینی,برق,سیگنالها و سیستمها,کتاب مرجع معرفی میکند پاورپوینت هم میدهد اما ا...,حضور و غیاب نمی کند,بله,منصفانه و هرچی خودت بگیری,10.0,10.0,10.0,8.0,10.0,10.0,ایمیل ایشان بود که در سایت نیز هست.,مهر 99,استاد خوبی هستند فقط امتحان های سختی میگیرن که...,0.9579,Positive
9,علی رضا ایوانی,مهندسی_مواد,درس شکل پذیری فلزات و شکست,فایل پاورپوینت یا پی دی اف می دهد,حضور مهم نیست اما تاثیر مثبت دارد,بله,منصفانه و هرچی خودت بگیری,10.0,10.0,10.0,10.0,10.0,10.0,بله گروه مجازی دارن,بهمن 99,سعی کنید سر کلاس خوب گوش بدید به مطالب,0.0000,Neutral
